# Analisis dan Penyesuaian Model Bahasa Besar (LLM) untuk Klasifikasi Ulasan dan Ringkasan Transkrip Menggunakan LangChain dan Replicate

## Deskripsi Proyek
Proyek ini adalah eksperimen Python yang menggunakan pustaka LangChain untuk berinteraksi dengan model bahasa besar (LLM) dari Replicate, khususnya model ibm-granite/granite-3.3-8b-instruct. Proyek ini berfokus pada dua tugas pemrosesan bahasa alami (NLP):
* Klasifikasi Sentimen dan Area Fokus: Menganalisis ulasan pengguna dan mengklasifikasikannya sebagai positif, negatif, atau campuran (mixed), sambil mengidentifikasi area fokus utama seperti UI/UX, Performa, Dukungan Pelanggan, Harga, atau Fitur.
* Ringkasan Transkrip: Meringkas transkrip rapat untuk mengekstrak informasi penting seperti keputusan utama, item tindakan (action items), dan tenggat waktu (deadlines).

Proyek ini mengeksplorasi bagaimana penyesuaian parameter model, seperti top_k, top_p, max_tokens, dan repetition_penalty, dapat memengaruhi kualitas dan format keluaran dari model LLM untuk kedua tugas tersebut.

## Install and import necessary libraries

In [ ]:
#!pip install langchain_community replicate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.6/48.6 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.2/45.2 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 3.2 MB/s eta 0:00:00


In [ ]:
from langchain_community.llms import Replicate
import os
from google.colab import userdata

In [ ]:
# Set the API token
# Replace 'REPLICATE_API_TOKEN' with the name of your Colab secret
api_token = userdata.get('REPLICATE_API_TOKEN')
os.environ["REPLICATE_API_TOKEN"] = api_token

In [ ]:
# Model setup
model = "ibm-granite/granite-3.3-8b-instruct"
output = Replicate(
    model=model,
    replicate_api_token=api_token,
)

## Test initial classification output with default parameters


In [ ]:
# Define the user reviews
user_reviews = [
    "The new software update is fast and stable, but the user interface is confusing.",
    "Customer support was excellent and solved my issue quickly, but the subscription price is too high.",
    "I love the new dark mode feature, but the app crashes every time I try to upload a file."
]

In [ ]:
# Refine the prompt to include reviews
reviews_text = "\n".join([f"Review {i+1}: {review}" for i, review in enumerate(user_reviews)])

In [ ]:
# Set model parameters for prompting with default values
parameters_default = {
    "top_k": 0,
    "top_p": 1.0,
    "max_tokens": 256,
    "min_tokens": 0,
    "random_seed": 1,
    "repetition_penalty": 1.0,
    "stopping_sequence": None
}


In [ ]:
# Add initial prompt
classification_prompt = f""":
Classify these reviews as positive, negative, or mixed, and tag relevant focus areas such as Performance, UI/UX, Customer Support, Pricing, or Features.
{reviews_text}"""

print("--- Initial Classification Output ---")
response_default = output.invoke(classification_prompt, parameters=parameters_default)
print(response_default)

--- Initial Classification Output ---
1. Review 1: 
- Classification: Mixed
- Focus Areas: Performance (Positive), UI/UX (Negative)

2. Review 2:
- Classification: Mixed
- Focus Areas: Customer Support (Positive), Pricing (Negative)

3. Review 3:
- Classification: Mixed
- Focus Areas: Features (Positive, mentioning dark mode), Performance/Stability (Negative, due to app crashes)


## Adjust parameters for better classification output

In [ ]:
# Refine multiple model parameter values
parameters_tuned = {
    "top_k": 5,
    "top_p": 0.8,
    "max_tokens": 50,  # Reduced for conciseness
    "min_tokens": 5,
    "random_seed": 1,
    "repetition_penalty": 1.2, # Increased to avoid redundancy
    "stopping_sequence": "\n\n" # Stop at a new paragraph
}

print("--- Tuned Classification Output ---")
response_tuned = output.invoke(classification_prompt, parameters=parameters_tuned)
print(response_tuned)

--- Tuned Classification Output ---
1. Review 1: 
- Classification: Mixed
- Focus Areas: Performance (Positive - fast and stable), UI/UX (Negative - confusing)

2. Review 2:
- Classification: Mixed
- Focus Areas: Customer Support (Positive - excellent and solved the issue quickly), Pricing (Negative - too high)

3. Review 3:
- Classification: Mixed
- Focus Areas: Features (Positive - love the dark mode), Performance/Bug (Negative - app crashes during file upload)


## Test initial summarization output with default parameters

In [ ]:
# Define the meeting transcript
meeting_transcript = """
The team discussed the upcoming product launch for "Project Phoenix". Key decisions included setting the launch date for November 1st and targeting a 15% market share in the first six months. The marketing team presented the campaign plan, which includes a social media blitz and a series of webinars. Concerns were raised about the beta testing feedback, particularly a bug in the file synchronization feature. It was decided that the engineering team would prioritize fixing this bug by the end of next week. The design team also requested more resources to finalize the marketing collateral, and it was agreed to reallocate a portion of the project budget to support them.
"""

In [ ]:
# Refine the prompt for summarization
summarization_prompt = f"""
Summarize the following meeting transcript, highlighting key decisions, action items, and deadlines.
Transcript: {meeting_transcript}
"""

print("--- Initial Summarization Output ---")
response_sum_default = output.invoke(summarization_prompt, parameters=parameters_default)
print(response_sum_default)

--- Initial Summarization Output ---
**Meeting Summary:**

**Key Decisions:**

1. Project Phoenix launch date is set for November 1st.
2. The marketing team's campaign plan, featuring a social media campaign and webinars, was approved.
3. A decision was made to aim for a 15% market share within the first six months post-launch.
4. The engineering team will prioritize resolving the critical bug in the file synchronization feature by the end of the following week.
5. Additional resources will be reallocated from the project budget to assist the design team in finalizing marketing collateral.

**Action Items:**

1. Engineering team: Fix the file synchronization bug by the end of next week.
2. Design team: Utilize the reallocation of project resources to finalize marketing materials.
3. Marketing team: Execute the planned social media blitz and series of webinars as per the approved campaign strategy.

**Deadlines:**

1. Bug fix in file synchronization feature: End of next week.
2. Finaliz

## Adjust parameters for better summarization output

In [ ]:
# Refine multiple parameter values for summarization
parameters_sum_tuned = {
    "top_k": 10,
    "top_p": 0.9,
    "max_tokens": 100, # Set a specific max length for a concise summary
    "min_tokens": 20,
    "random_seed": None, # Allow for varied output
    "repetition_penalty": 1.5, # Penalize repetitive phrases
    "stopping_sequence": "##"
}

print("--- Tuned Summarization Output ---")
response_sum_tuned = output.invoke(summarization_prompt, parameters=parameters_sum_tuned)
print(response_sum_tuned)

--- Tuned Summarization Output ---
**Summary of Meeting:**

*Launch Date and Market Share Target:*
- The team decided to launch "Project Phoenix" on November 1st.
- A goal of acquiring 15% market share within the first six months was established.

*Marketing Campaign Plan:*
- The marketing team unveiled a campaign plan comprising a social media campaign and a series of webinars.

*Bug Fix and Beta Testing Feedback:*
- Concerns were voiced regarding a file synchronization bug identified during beta testing.
- It was resolved that the engineering team would prioritize resolving this bug by the end of the following week.

*Resource Allocation for Design Team:*
- The design team expressed the need for additional resources to finalize marketing collateral.
- The team agreed to reallocate part of the project budget to support the design team's request.

**Key Decisions:**
1. Launch date set for November 1st.
2. Market share target set at 15% for the first six months post-launch.
3. Engineeri

## Kesimpulan

1. Penyesuaian parameter model adalah kunci untuk mengoptimalkan kinerja LLM pada tugas-tugas spesifik.

2. Dengan parameter yang tepat, model dapat menghasilkan keluaran yang lebih akurat dan terstruktur sesuai kebutuhan.